In [0]:
# Install required library to read Excel files
%pip install openpyxl


In [0]:
# Import libraries
import pandas as pd

# Load the dataset
# The file has 2 sheets - Year 2009-2010 and Year 2010-2011
# We'll load both and combine them

df1 = pd.read_excel(
    "/Volumes/workspace/default/retail_data/online_retail_II.xlsx",
    sheet_name="Year 2009-2010"
)

df2 = pd.read_excel(
    "/Volumes/workspace/default/retail_data/online_retail_II.xlsx",
    sheet_name="Year 2010-2011"
)

# Combine both sheets into one dataframe
df = pd.concat([df1, df2], ignore_index=True)

print(f"Total rows loaded: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names: {df.columns.tolist()}")

In [0]:
# First look at the data
print("=== FIRST 5 ROWS ===")
print(df.head())

print("\n=== DATA TYPES ===")
print(df.dtypes)

print("\n=== BASIC STATISTICS ===")
print(df.describe())

print("\n=== MISSING VALUES ===")
print(df.isnull().sum())
print(f"\nMissing values as percentage:")
print((df.isnull().sum() / len(df) * 100).round(2))

In [0]:
# DATA CLEANING - SEPARATING LOYALTY MEMBERS VS GUESTS/WALKINS
# ------------------------------------------------------------

# --- STEP 1: SPLIT GUESTS FROM MEMBERS ---
# Separate rows where Customer ID is missing into guest dataset
# .copy() creates a fresh copy so changes don't affect the main df
df_guest = df[df['Customer ID'].isna()].copy()
df_guest['Customer ID'] = 'Guest'
print(f"Guest/Walk-in transactions: {len(df_guest):,}")

# Known members — rows where Customer ID is NOT missing
df_memb = df[~df['Customer ID'].isna()].copy()
print(f"Member transactions: {len(df_memb):,}")

# --- STEP 2: REMOVE CANCELLED ORDERS ---
# Invoices starting with 'C' are cancellations
# These are not real completed purchases
df_memb = df_memb[~df_memb['Invoice'].astype(str).str.startswith('C')]
df_guest = df_guest[~df_guest['Invoice'].astype(str).str.startswith('C')]
print(f"\nAfter removing cancellations:")
print(f"Members: {len(df_memb):,} rows")
print(f"Guest/Walk-in: {len(df_guest):,} rows")

# --- STEP 3: REMOVE INVALID QUANTITIES AND PRICES ---
# Negative quantities = returns/refunds (not purchases)
# Zero prices = free samples or data entry errors
# We only keep rows where BOTH are greater than 0
df_memb = df_memb[(df_memb['Quantity'] > 0) & (df_memb['Price'] > 0)]
df_guest = df_guest[(df_guest['Quantity'] > 0) & (df_guest['Price'] > 0)]

# --- STEP 4: REMOVE MISSING DESCRIPTIONS ---
# Small number of missing descriptions — safe to remove
df_memb = df_memb.dropna(subset=['Description'])
df_guest = df_guest.dropna(subset=['Description'])

# --- STEP 5: FIX DATA TYPES ---
# Convert Customer ID to whole number — removes decimal e.g. 12345.0 → 12345
# Only for members since guests have a text label
df_memb['Customer ID'] = df_memb['Customer ID'].astype(int)

# --- STEP 6: CREATE TOTAL REVENUE COLUMN ---
# TotalRevenue = how much money each transaction line generated
# Quantity x Price = value of each line item
df_memb['TotalRevenue'] = df_memb['Quantity'] * df_memb['Price']
df_guest['TotalRevenue'] = df_guest['Quantity'] * df_guest['Price']

# --- STEP 7: SUMMARY ---
print(f"\n CLEANING COMPLETE")
print(f"Member transactions: {len(df_memb):,} rows")
print(f"Guest/Walk-in transactions: {len(df_guest):,} rows")
print(f"\nMember revenue: £{df_memb['TotalRevenue'].sum():,.2f}")
print(f"Guest/Walk-in revenue: £{df_guest['TotalRevenue'].sum():,.2f}")
print(f"Total revenue: £{(df_memb['TotalRevenue'].sum() + df_guest['TotalRevenue'].sum()):,.2f}")

In [0]:
# Final check — make sure our clean data looks right
print("=== MEMBER DATA SAMPLE ===")
print(df_memb.head())

print("\n=== GUEST DATA SAMPLE ===")
print(df_guest.head())

print("\n=== ANY REMAINING MISSING VALUES IN MEMBERS? ===")
print(df_memb.isnull().sum())

print("\n=== NEGATIVE VALUES REMAINING? ===")
print(f"Negative quantities: {len(df_memb[df_memb['Quantity'] < 0])}")
print(f"Zero prices: {len(df_memb[df_memb['Price'] == 0])}")